## 01 &middot; Reference solver: Baseline in NumPy

Given a 1D shallow-water wave packet, let's define a function that
advances it one timestep. Throughout this tutorial, we will effectively
rewrite the same function **five different ways** and examine our results.
To start, this notebook locks down the baseline reference implementation
in NumPy against which we will compare other tools and programming
models.

### Table of contents

1. [What we're solving](#sec1)
2. [The function we're replacing](#sec2)
3. [Run the NumPy baseline](#sec3)
4. [Profiling where we spend time](#sec4)
5. [What each later notebook replaces](#sec5)

### <a id="sec1"></a>1. What we're solving

The 1D shallow-water equations track water height $h$ and momentum $hu$
along a channel:

$$
\frac{\partial h}{\partial t} + \frac{\partial (hu)}{\partial x} = 0,
\qquad
\frac{\partial (hu)}{\partial t} + \frac{\partial}{\partial x}\!\left( \tfrac{(hu)^2}{h} + \tfrac{1}{2} g h^2 \right) = 0.
$$

`swe_core.step_numpy` advances both by one timestep using forward Euler
in time and a Rusanov flux at each cell interface. The reference implementation lives in `swe_core.py`. For the scope of this tutorial, what matters
is the **shape** of the operation:

- Read the two neighbours of each cell.
- Compute a flux at each face from the two states it sits between.
- Update each cell by `(Δt/Δx)` times the difference of its two adjacent fluxes.

We're going to re-implement the function that performs these three steps with different libraries and programming models, and explore their strengths and limitations.

### <a id="sec2"></a>2. The function we're replacing

`swe_core.step_numpy(h, hu, dx, dt, g)` is a concise ~15-line vectorised
NumPy implementation:

```py
def step_numpy(h, hu, dx, dt, g=g, tol=DRY_TOL):
    """One forward-Euler step with Rusanov flux. Returns (h_new, hu_new)."""
    # Left/right states at every interface i+½  (length N+1).
    hL, hR   = h[:-1],  h[1:]
    huL, huR = hu[:-1], hu[1:]

    # Velocities and wave speeds at the face.
    h_safe_L = np.maximum(hL, tol)
    h_safe_R = np.maximum(hR, tol)
    uL, uR = huL / h_safe_L,        huR / h_safe_R
    cL, cR = np.sqrt(g * h_safe_L), np.sqrt(g * h_safe_R)
    a      = np.maximum(np.abs(uL) + cL, np.abs(uR) + cR)

    # Rusanov interface flux: average of physical fluxes − stabilising diffusion.
    F_h  = 0.5 * (huL + huR) - 0.5 * a * (hR  - hL)
    F_hu = 0.5 * (huL*uL + 0.5*g*hL*hL + huR*uR + 0.5*g*hR*hR) - 0.5 * a * (huR - huL)

    # Divergence: interior cells only; ghost cells unchanged.
    h_new, hu_new = h.copy(), hu.copy()
    h_new[1:-1]   = h[1:-1]  - (dt / dx) * (F_h[1:]  - F_h[:-1])
    hu_new[1:-1]  = hu[1:-1] - (dt / dx) * (F_hu[1:] - F_hu[:-1])
    return h_new, hu_new
```

The body is the three blocks from section 1 in order: **read** the per-face
interface states, **compute** the Rusanov flux, **update** each
interior cell from the divergence of those fluxes.

The rest of the bookkeeping: the initial and boundary conditions, time loop, and validation functions remain the same.


### <a id="sec3"></a>3. Run the NumPy baseline

Let's run the full simulation, time it, and save one row to `timings.json`. Each
later notebook writes its own row to the same file and reports the same
acceptance metric.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import swe_core

N        = 16384
L        = 10.0
H0       = 1.0
AMP      = 0.1
SIG      = 0.5
CFL      = 0.4
G        = 9.81
N_STEPS  = 1000
dx       = L / N
DT       = swe_core.fixed_dt(H0 + AMP, dx, cfl=CFL, g=G)

def run_numpy():
    h, hu = swe_core.bump_ic(N, L=L, h0=H0, amplitude=AMP, sigma=SIG)
    for _ in range(N_STEPS):
        swe_core.apply_bc_reflective(h, hu)
        h, hu = swe_core.step_numpy(h, hu, dx, DT, g=G)
    return h, hu

result = swe_core.timed_run(run_numpy, warmup=2, repeats=5, label='01_numpy')
cells_per_s = N * N_STEPS / result['median_s']
print(f"NumPy   {N} cells x {N_STEPS} steps : median {result['median_s']*1000:6.1f} ms")
print(f"  throughput                          : {cells_per_s/1e6:6.1f} M cell-updates/s")

swe_core.save_timing(result, grid_str=f'N={N}', tool='numpy', hardware='cpu',
                     dtype='float64', steps=N_STEPS)

# Un-timed pass for the multi-snapshot plot. Runs longer than the timed
# loop above so the two pulses separate clearly; only snapshots are kept.
PLOT_STEPS = 2000
snap_steps = {0, 250, 500, 1000, 2000}
h, hu = swe_core.bump_ic(N, L=L, h0=H0, amplitude=AMP, sigma=SIG)
snaps = {0: h[1:-1].copy()}
for step in range(1, PLOT_STEPS + 1):
    swe_core.apply_bc_reflective(h, hu)
    h, hu = swe_core.step_numpy(h, hu, dx, DT, g=G)
    if step in snap_steps:
        snaps[step] = h[1:-1].copy()

xs = (np.arange(N) + 0.5) * dx
fig, ax = plt.subplots(figsize=(9, 3.4))
colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(snaps)))
for (step, h_snap), color in zip(sorted(snaps.items()), colors):
    label = f't = {step*DT:.3f} s' + ('  (IC)' if step == 0 else '')
    ax.plot(xs, h_snap, color=color, label=label, linewidth=1.6)
ax.set_xlabel('x  [m]'); ax.set_ylabel('h  [m]')
ax.set_title('Gaussian bump propagation — water height at successive times')
ax.legend(loc='upper right', fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### <a id="sec4"></a>4. Profiling where we spend time

Each iteration of `run_numpy` does two pieces of work: apply boundary
conditions, then compute the flux and update the two variables. How
does that cost split, and how does the split change with `N`?

The cell below times each component separately at
`N ∈ {1024, 4096, 16384}`.

In [ ]:
def time_components(N_local, n_steps=200):
    L_local  = L
    dx_local = L_local / N_local
    dt_local = swe_core.fixed_dt(H0 + AMP, dx_local, cfl=CFL, g=G)

    # Warm run
    h, hu = swe_core.bump_ic(N_local, L=L_local, h0=H0, amplitude=AMP, sigma=SIG)
    for _ in range(2):
        swe_core.apply_bc_reflective(h, hu)
        h, hu = swe_core.step_numpy(h, hu, dx_local, dt_local, g=G)

    # Measured pass — fresh IC.
    h, hu = swe_core.bump_ic(N_local, L=L_local, h0=H0, amplitude=AMP, sigma=SIG)
    t_bc = t_step = 0.0
    for _ in range(n_steps):
        t0 = time.perf_counter()
        swe_core.apply_bc_reflective(h, hu)
        t1 = time.perf_counter()
        h, hu = swe_core.step_numpy(h, hu, dx_local, dt_local, g=G)
        t2 = time.perf_counter()
        t_bc   += t1 - t0
        t_step += t2 - t1
    return t_bc / n_steps, t_step / n_steps

print(f"{'N':>6}  {'BC [µs]':>10}  {'step [µs]':>12}  {'step/total':>12}  {'M cells/s':>12}")
for N_b in [1024, 4096, 16384]:
    bc, step = time_components(N_b)
    pct  = step / (bc + step) * 100
    rate = N_b / (bc + step) / 1e6
    print(f"{N_b:6d}  {bc*1e6:10.2f}  {step*1e6:12.2f}  {pct:11.1f}%  {rate:12.1f}")

Takeaways:

- **Boundary condition roughly constant in `N`** &mdash; touches only ghost cells.
- **Step time scales linearly in `N`** &mdash; stencil work is O(N).
- **At large `N`, the step dominates entirely**, so later notebooks replace the step kernel, not the boundary handling.

### <a id="sec5"></a>5. What each later notebook replaces

Each later notebook implements `step_<tool>(h, hu, dx, dt) -> (h_new, hu_new)`
on a different stack, then checks the result against `step_numpy`.

The acceptance gate across the board:

$\max_i |h_{\text{tool}}[i] - h_{\text{numpy}}[i]| < \mathrm{TOL}$

where $\mathrm{TOL} = 10^{-12}$ for float64 and $10^{-4}$ for float32.